In [25]:
# DiD

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load data 

data = pd.read_stata("2_analysisFirstStageAirQuality.dta")

# Convert dates 

data['dateNum'] = pd.to_datetime(data['dateNum'], errors='coerce')
data['ezpass_date'] = pd.to_datetime(data['ezpass_date'], errors='coerce')

# Extract time variables 

data['year'] = data['dateNum'].dt.year
data['month'] = data['dateNum'].dt.month
data['dow'] = data['dateNum'].dt.dayofweek

# Restrict to NJ only 

data['state'] = (data['monitorID'] // 10000000).astype(int)
data = data[data['state'] == 34]

# Treatment and DiD variables

data['near_toll'] = (data['minDistanceMonitor'] < 2).astype(int)  # Treatment: <2km
data['post'] = (data['dateNum'] >= data['ezpass_date']).astype(int)
data['did'] = data['near_toll'] * data['post']

# Log outcome 

data['log_NO2'] = np.log(data['dailyMeanno2'] + 1e-6)  # avoid log(0)

# Monitor-specific linear trend 

data['min_year'] = data.groupby('monitorID')['year'].transform('min')
data['monitor_trend'] = data['year'] - data['min_year']

# Clean data 

df = data.dropna(subset=['log_NO2','near_toll','post','did','distanceMonitor',
                         'year','month','dow','toll_id','monitor_trend'])

# Run DiD regression 

formula = 'log_NO2 ~ near_toll + post + did + distanceMonitor + C(year) + C(month) + C(dow) + C(toll_id) + C(monitorID):monitor_trend'

model = smf.ols(formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['dateNum']})

# Extract results 

coef = model.params['did']
se = model.bse['did']
n_obs = df.shape[0]
sig = '**' if abs(coef/se) > 1.96 else '*' if abs(coef/se) > 1.645 else ''

# Print results 

print("NO2 All Controls (DiD)")
print(f"Coef: {coef:.3f}")
print(f"SE: {se:.3f}")
print(f"N: {n_obs}")
print(f"Significance: {sig}")
print(df.columns.tolist())

NO2 All Controls (DiD)
Coef: -0.126
SE: 0.020
N: 19356
Significance: **
['monitorID', 'dateNum', 'dailyMeanco', 'dailyMaxco', 'pollReadingsco', 'dailyMeanno2', 'dailyMaxno2', 'pollReadingsno2', 'dailyMeannox', 'dailyMaxnox', 'pollReadingsnox', 'dailyMeano3', 'dailyMaxo3', 'pollReadingso3', 'dailyMeanpm10', 'dailyMaxpm10', 'pollReadingspm10', 'dailyMeanso2', 'dailyMaxso2', 'pollReadingsso2', 'dailyMeanvoc', 'dailyMaxvoc', 'pollReadingsvoc', 'latitude', 'longitude', 'gridNumber', 'tMin', 'tMax', 'prec', 'year', 'count', '_merge', 'join', 'ezpass_date', 'longitudeToll', 'latitudeToll', 'toll_id', 'distanceMonitor', 'minDistanceMonitor', 'pollutionControl', 'month', 'dow', 'state', 'near_toll', 'post', 'did', 'log_NO2', 'min_year', 'monitor_trend']


In [30]:
# Cross-moment functions

def compute_ratio(Z, D, deg=2):
    var_u = np.mean(Z * D)
    sign = np.sign(var_u)
    diff_normal_D = np.mean(D ** deg * Z) - deg * var_u * np.mean(D ** (deg - 1))
    diff_normal_Z = np.mean(Z ** deg * D) - deg * var_u * np.mean(Z ** (deg - 1))
    epsilon = 1e-8
    alpha_sq = diff_normal_D / (diff_normal_Z + epsilon)
    if alpha_sq < 0:
        alpha_sq = -(abs(alpha_sq) ** (1 / (deg - 1)))
    else:
        alpha_sq = alpha_sq ** (1 / (deg - 1))
    alpha_sq = abs(alpha_sq) * sign
    return alpha_sq

def get_beta(Z, D, Y, deg=2):
    denominator = 0
    while denominator == 0:
        alpha_sq = compute_ratio(Z, D, deg)
        numerator = np.mean(D * Y) - alpha_sq * np.mean(Y * Z)
        denominator = np.mean(D * D) - alpha_sq * np.mean(D * Z)
        deg += 1
    return numerator / denominator

In [67]:
# Cross-moment

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# Load data 

data = pd.read_stata("2_analysisFirstStageAirQuality.dta")

# Convert dates

data['dateNum'] = pd.to_datetime(data['dateNum'], errors='coerce')
data['ezpass_date'] = pd.to_datetime(data['ezpass_date'], errors='coerce')

# Extract time variables 

data['year'] = data['dateNum'].dt.year
data['month'] = data['dateNum'].dt.month
data['dow'] = data['dateNum'].dt.dayofweek

# Restrict to NJ only 

data['state'] = (data['monitorID'] // 10000000).astype(int)
data = data[data['state'] == 34]

# Treatment and DiD variables 

data['near_toll'] = (data['minDistanceMonitor'] < 2).astype(int)
data['post'] = (data['dateNum'] >= data['ezpass_date']).astype(int)
data['did'] = data['near_toll'] * data['post']

# Log outcome 

data['log_NO2'] = np.log(data['dailyMeanno2'] + 1e-6)

# Monitor-specific linear trend 

data['min_year'] = data.groupby('monitorID')['year'].transform('min')
data['monitor_trend'] = data['year'] - data['min_year']

# Clean data 

df = data.dropna(subset=['log_NO2','near_toll','post','did','distanceMonitor',
                         'year','month','dow','toll_id','monitor_trend'])


# Cross-moment functions

def compute_ratio(Z, D, deg=2):
    var_u = np.mean(Z * D)
    sign = np.sign(var_u)
    diff_normal_D = np.mean(D ** deg * Z) - deg * var_u * np.mean(D ** (deg - 1))
    diff_normal_Z = np.mean(Z ** deg * D) - deg * var_u * np.mean(Z ** (deg - 1))
    epsilon = 1e-8
    alpha_sq = diff_normal_D / (diff_normal_Z + epsilon)
    if alpha_sq < 0:
        alpha_sq = -(abs(alpha_sq) ** (1 / (deg - 1)))
    else:
        alpha_sq = alpha_sq ** (1 / (deg - 1))
    alpha_sq = abs(alpha_sq) * sign
    return alpha_sq

def get_beta(Z, D, Y, deg=2):
    denominator = 0
    while denominator == 0:
        alpha_sq = compute_ratio(Z, D, deg)
        numerator = np.mean(D * Y) - alpha_sq * np.mean(Y * Z)
        denominator = np.mean(D * D) - alpha_sq * np.mean(D * Z)
        deg += 1
    return numerator / denominator


# Prepare covariates for residualization

covariates = ['distanceMonitor', 'year', 'month', 'dow', 'toll_id', 'monitor_trend']

# Pre-period: before EZPass

df_pre = df[df['post'] == 0]

# Post-period: after EZPass

df_post = df[df['post'] == 1]

# Build X matrices including treatment and DiD

X_pre = df_pre[covariates + ['near_toll', 'did']].to_numpy()
X_post = df_post[covariates + ['near_toll', 'did']].to_numpy()

Y_pre = df_pre['log_NO2'].to_numpy()
Y_post = df_post['log_NO2'].to_numpy()

# Fit regression on all covariates including treatment

reg = LinearRegression().fit(X_pre, Y_pre)

# Pre-period treatment (near_toll) stacked with post-period

D = np.concatenate([df_pre['near_toll'].to_numpy(), df_post['near_toll'].to_numpy()]).astype(float)

# Stack outcomes

Y = np.concatenate([Y_pre, Y_post])


# Residualize

Z = Y - X[:, :-1] @ reg.coef_[:-1]

# Convert to float before centering

Z = Z.astype(float)
Y = Y.astype(float)

# Center variables

Z -= np.mean(Z)
D -= np.mean(D)
Y -= np.mean(Y)

# Cross-moment loop

max_deg = 12
betas_est = np.zeros(max_deg-2)

for deg in range(2, max_deg):
    beta = get_beta(Z, D, Y, deg)
    betas_est[deg-2] = beta

print("Cross-moment beta estimate (NO2):", betas_est[0])

Cross-moment beta estimate (NO2): 0.9108311197820029
